# P4: Label Denoising with Confident Learning (Provenance)

Stage 3 of the dataset pipeline. The Stage-2 expansion knowingly poisoned
the no-eruption class: real eruptions missing from the ground-truth catalogs
looked like "false positives" and were filed as background. Confident
Learning (Northcutt et al. 2021, `cleanlab`) finds them: train 5-fold
cross-validated models, collect *out-of-sample* predictions for every image,
and flag images whose predictions consistently contradict their labels.

> **Staged manual protocol (output already provided).** This notebook is one
> *pass*. The published denoising ran the notebook repeatedly with a manual
> review between passes: first requiring 5-of-5 iteration consensus,
> manually verifying and re-filing the flagged events, then re-running at
> 4-of-5 on the cleaned data, then 3-of-5. This is why the consensus test
> below is an exact match (`==`), not `>=`. Across the three passes, 1,288
> images (14 large-eruption events + 406 small) moved from "no eruption" to
> their true classes. The shipped `Dataset_2014_2015` is the final state;
> re-running on it should flag (almost) nothing.

In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
import os, glob, shutil, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers, models, callbacks, optimizers
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils import class_weight
from cleanlab.filter import find_label_issues

NUM_ITERATIONS = 5      # independent cross-validation repetitions
CONSENSUS_INDEX = 5     # pass 1: flagged in exactly 5/5; re-run at 4, then 3
IMG_H, IMG_W = config.IMG_HEIGHT, config.IMG_WIDTH
BATCH_SIZE = 128
N_SPLITS = 5
CLEANING_EPOCHS = 1000
DRY_RUN = True          # True: report flagged events only; False: move them to review

REVIEW_DIR = config.OUTPUT_DIR / "Review_Cleanlab"


## 1. Index, group into events, shuffle

Grouping mirrors training: eruptions at 10 s, background at 300 s. The
"Group" condition of the K-fold keeps each event's frames in one fold; the
"Stratified" condition keeps class proportions equal across folds.

In [ ]:
records = []
for label_idx, folder in config.CLASS_FOLDERS.items():
    for f in glob.glob(str(config.DATASET_DIR / folder / "*.jpg")):
        records.append({"path": f, "label": label_idx})
df = pd.DataFrame(records)

def get_timestamp(path):
    parts = os.path.splitext(os.path.basename(path))[0].split("_")
    return pd.to_datetime(parts[0] + " " + parts[1], format="%Y-%m-%d %H%M%S", errors="coerce")

df["timestamp"] = df["path"].map(get_timestamp)
df = df.sort_values(["label", "timestamp"]).reset_index(drop=True)
df["dt"] = df.groupby("label")["timestamp"].diff()

gap_eruption   = pd.Timedelta(seconds=config.EVENT_GAP_SEC)
gap_background = pd.Timedelta(seconds=config.NO_ERUPTION_GROUPING_SEC)
new_event = (((df["label"].isin([1, 2])) & (df["dt"] > gap_eruption))
             | ((df["label"] == 0) & (df["dt"] > gap_background))
             | df["dt"].isna())
df["event_id"] = df["label"].astype(str) + "_" + new_event.groupby(df["label"]).cumsum().astype(str)
df = df.sample(frac=1, random_state=config.RANDOM_SEED).reset_index(drop=True)
print(f"{len(df):,} images, {df['event_id'].nunique():,} events")


## 2. Cross-validated out-of-sample predictions, repeated 5 times

Each iteration: a fresh 5-fold `StratifiedGroupKFold` (different shuffle
seed), a fresh model per fold trained with sparse categorical crossentropy
and balanced class weights, and predictions collected on each held-out
fold, so every image gets one out-of-sample prediction per iteration. `cleanlab`
then ranks label issues; an image collects one "vote" per iteration that
flags it. Runs 25 model trainings, a multi-day GPU process.

In [ ]:
def make_ds(sub, training):
    ds = tf.data.Dataset.from_tensor_slices((sub["path"].values, sub["label"].values.astype("int32")))
    def load(p, l):
        img = tf.image.decode_jpeg(tf.io.read_file(p), channels=3)
        return tf.image.resize(img, [IMG_H, IMG_W]), l
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(5000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

class RestoreBestOnLRReduce(callbacks.Callback):
    """In-memory variant: snapshot best-val weights; restore them when the LR drops."""
    def on_train_begin(self, logs=None):
        self.best_weights, self.best_loss = self.model.get_weights(), np.inf
        self.prev_lr = float(self.model.optimizer.learning_rate)
    def on_epoch_end(self, epoch, logs=None):
        vl = (logs or {}).get("val_loss")
        if vl is not None and vl < self.best_loss:
            self.best_loss, self.best_weights = vl, self.model.get_weights()
        lr = float(self.model.optimizer.learning_rate)
        if lr < self.prev_lr - 1e-9:
            self.model.set_weights(self.best_weights)
        self.prev_lr = lr

def create_model():
    m = models.Sequential([
        layers.Input(shape=(IMG_H, IMG_W, 3)),
        layers.Cropping2D(cropping=((30, 30), (40, 40))),
        layers.RandomRotation(0.1),
        layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        layers.RandomZoom(height_factor=(-0.2, 0.1), width_factor=(-0.2, 0.1)),
        layers.RandomContrast(0.2), layers.RandomBrightness(0.1),
        layers.Rescaling(1.0 / 255),
        layers.Conv2D(16, 3, padding="same", kernel_initializer="he_normal"),
        layers.LeakyReLU(negative_slope=0.1), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal"),
        layers.LeakyReLU(negative_slope=0.1), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal"),
        layers.LeakyReLU(negative_slope=0.1), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, kernel_initializer="he_normal"),
        layers.LeakyReLU(negative_slope=0.01),
        layers.Dense(3, activation="softmax"),
    ])
    m.compile(optimizer=optimizers.Adam(learning_rate=0.001, clipnorm=0.1),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

vote_counts = np.zeros(len(df))
for i in range(NUM_ITERATIONS):
    print(f"\n=== ITERATION {i + 1}/{NUM_ITERATIONS} ===")
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=config.RANDOM_SEED + i)
    oos_probs = np.zeros((len(df), 3))
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df["label"], df["event_id"])):
        print(f"  fold {fold + 1}/{N_SPLITS}")
        train_sub, val_sub = df.iloc[train_idx], df.iloc[val_idx]
        weights = class_weight.compute_class_weight(
            class_weight="balanced", classes=np.unique(train_sub["label"]), y=train_sub["label"])
        tf.keras.backend.clear_session()
        model = create_model()
        model.fit(make_ds(train_sub, True), validation_data=make_ds(val_sub, False),
                  epochs=CLEANING_EPOCHS, class_weight=dict(enumerate(weights)), verbose=2,
                  callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=36,
                                                     restore_best_weights=True),
                             callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                         patience=3, min_lr=1e-6),
                             RestoreBestOnLRReduce()])
        oos_probs[val_idx] = model.predict(make_ds(val_sub, False), verbose=0)
    oos_probs[np.isnan(oos_probs)] = 1 / 3
    issues = find_label_issues(labels=df["label"].values, pred_probs=oos_probs,
                               return_indices_ranked_by="self_confidence")
    vote_counts[issues] += 1
    print(f"  iteration {i + 1}: {len(issues):,} candidate label issues")


## 3. Consensus selection and human review

Only images flagged in exactly `CONSENSUS_INDEX` of the iterations are
selected: the exact match implements the staged protocol described in the
header (run at 5, review, re-file, re-run at 4, then 3). Flagged events go
to a review folder (or are just listed under `DRY_RUN`); the human decides
each event's true class and moves it, then the next pass begins.

In [ ]:
consensus = np.where(vote_counts == CONSENSUS_INDEX)[0]
print(f"Flagged at least once: {int(np.count_nonzero(vote_counts)):,}")
print(f"Consensus (exactly {CONSENSUS_INDEX}/{NUM_ITERATIONS}): {len(consensus):,} images")

issue_events = df.iloc[consensus]["event_id"].unique()
print(f"Distinct flagged events: {len(issue_events)}")
for eid in issue_events:
    rows = df[df["event_id"] == eid]
    folder = config.CLASS_FOLDERS[int(rows.iloc[0]["label"])]
    if DRY_RUN:
        print(f"  [would review] event {eid} ({folder}, {len(rows)} frames, "
              f"first: {os.path.basename(rows.iloc[0]['path'])})")
    else:
        dest = REVIEW_DIR / folder
        dest.mkdir(parents=True, exist_ok=True)
        for _, row in rows.iterrows():
            shutil.move(row["path"], dest / os.path.basename(row["path"]))
print("\nAfter human review + re-filing, lower CONSENSUS_INDEX and re-run "
      "(published protocol: 5 -> 4 -> 3; total moved: 1,288 images).")
